# 03 — Carga y visualización de ECG desde VitalDB

Descarga de la señal ECG cruda para un subconjunto reducido de `case_id` usando la librería `vitaldb` y visualización de tramos.

**Advertencias**

- La descarga depende de conexión a internet y puede ser lenta.
- La señal se almacena localmente en `data/raw/vitaldb_waveforms/` (excluida del repositorio).
- Verifica el nombre del canal ECG real en VitalDB antes de descargar todos los casos.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.data_loading import load_metadata
from src.download import load_ecg_from_vitaldb, save_ecg_npy
from src.utils import ensure_dir, get_logger

logger = get_logger("nb03")

## 2. Selección del subconjunto de casos

Definir manualmente unos pocos `case_id` para probar la carga. Ajustar al criterio del usuario.

In [ ]:
metadata = load_metadata()

# Tomar un puñado de case_id presentes en metadata para probar la descarga.
# La selección se hace por posición, NO al azar, para que sea reproducible.
sample_case_ids = metadata[config.CASE_ID_COLUMN].head(3).tolist()
print("case_ids a cargar:", sample_case_ids)

## 3. Descarga y persistencia local

In [ ]:
ensure_dir(config.VITALDB_WAVEFORMS_DIR)

ecg_by_case = {}
for cid in sample_case_ids:
    cached = config.VITALDB_WAVEFORMS_DIR / f"case_{cid}.npy"
    if cached.exists():
        logger.info("Cargando desde caché: %s", cached)
        signal = np.load(cached)
    else:
        logger.info("Descargando desde VitalDB: case_id=%s", cid)
        signal = load_ecg_from_vitaldb(
            cid,
            track_name=config.DEFAULT_ECG_TRACK_NAME,
            sampling_rate_hz=config.DEFAULT_ECG_FS_HZ,
        )
        save_ecg_npy(signal, cid)
    ecg_by_case[cid] = signal
    print(f"case_id={cid} | shape={signal.shape}")

## 4. Visualización de un tramo

In [ ]:
fs = config.DEFAULT_ECG_FS_HZ
window_seconds = 10

for cid, signal in ecg_by_case.items():
    n = min(int(fs * window_seconds), signal.shape[0])
    t = np.arange(n) / fs
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(t, signal[:n], linewidth=0.8)
    ax.set_title(f"ECG | case_id={cid} | primeros {window_seconds} s")
    ax.set_xlabel("Tiempo (s)")
    ax.set_ylabel("Amplitud")
    plt.tight_layout()
    plt.show()

## 5. Próximos pasos

- Continuar con `04_windowing_and_feature_engineering.ipynb` para construir ventanas alrededor de los latidos.